# expA00_baseline: sitting/standing 分類ベースライン

## 実験概要
COCO形式のkeypoint・bboxデータから sitting / standing を判別する。  
ルールベース（v1/v2）と LightGBM の3アプローチを比較する。

## アプローチ比較

| アプローチ | 特徴量数 | CV Macro F1 | 備考 |
|-----------|---------|-------------|------|
| ルールベース v1 | 5 | ≈ 0.7172 | aspect ratioのif/elif順バグあり |
| ルールベース v2 | 8 | ≈ 0.7300 | バグ修正 + 特徴量追加 |
| LightGBM 5-fold | 22 | ≈ 0.7915 | StratifiedKFold, OOF評価 |

In [1]:
import json
import math
import pickle
import numpy as np
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import lightgbm as lgb

# COCO 17 keypoint indices
NOSE = 0
LEFT_EYE = 1
RIGHT_EYE = 2
LEFT_EAR = 3
RIGHT_EAR = 4
LEFT_SHOULDER = 5
RIGHT_SHOULDER = 6
LEFT_ELBOW = 7
RIGHT_ELBOW = 8
LEFT_WRIST = 9
RIGHT_WRIST = 10
LEFT_HIP = 11
RIGHT_HIP = 12
LEFT_KNEE = 13
RIGHT_KNEE = 14
LEFT_ANKLE = 15
RIGHT_ANKLE = 16

# 定数
SEED = 42
N_FOLDS = 5

# パス定義 (__file__ 非依存, Path.cwd() ベース)
# expA00_baseline (cwd) -> parents[0]=workspace -> parents[1]=repo root -> datasets
DATA_DIR = Path.cwd().parents[1] / "datasets"
RESULTS_DIR = Path.cwd() / "results"

print(f"DATA_DIR   : {DATA_DIR}")
print(f"RESULTS_DIR: {RESULTS_DIR}")
print(f"DATA_DIR exists: {DATA_DIR.exists()}")

DATA_DIR   : /Users/estyle-155/Documents/work_sample_test_starterRepository/datasets
RESULTS_DIR: /Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/expA00_baseline/results
DATA_DIR exists: True


In [2]:
# ============================================================
# Cell 3: 特徴量抽出関数 (src/features.py の内容)
# ============================================================

def _angle_between(p1, p2, p3):
    """3点 p1-p2-p3 のp2における角度(度)を計算"""
    v1 = np.array(p1) - np.array(p2)
    v2 = np.array(p3) - np.array(p2)
    cos_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-8)
    cos_angle = np.clip(cos_angle, -1.0, 1.0)
    return math.degrees(math.acos(cos_angle))


def extract_features(data: dict) -> dict:
    """入力dictから特徴量dictを抽出する (22特徴量)

    Args:
        data: bbox, bbox_confidence, keypoints, keypoint_scores を含むdict

    Returns:
        特徴量名をkey、値をvalueとするdict
    """
    bbox = data["bbox"]
    kps = np.array(data["keypoints"])    # (17, 2)
    scores = np.array(data["keypoint_scores"])  # (17,)
    bx, by, bw, bh = bbox["x"], bbox["y"], bbox["w"], bbox["h"]

    # bbox正規化キーポイント
    rel_kps = np.zeros_like(kps)
    rel_kps[:, 0] = (kps[:, 0] - bx) / (bw + 1e-8)
    rel_kps[:, 1] = (kps[:, 1] - by) / (bh + 1e-8)

    features = {}

    # 1. bbox aspect ratio (最も単純で強力)
    features["bbox_aspect_ratio"] = bh / (bw + 1e-8)

    # 2. hip-knee垂直距離 (最強シグナル d>1.1)
    features["left_hip_knee_vert"] = rel_kps[LEFT_KNEE, 1] - rel_kps[LEFT_HIP, 1]
    features["right_hip_knee_vert"] = rel_kps[RIGHT_KNEE, 1] - rel_kps[RIGHT_HIP, 1]

    # 3. hip-ankle垂直距離
    features["left_hip_ankle_vert"] = rel_kps[LEFT_ANKLE, 1] - rel_kps[LEFT_HIP, 1]
    features["right_hip_ankle_vert"] = rel_kps[RIGHT_ANKLE, 1] - rel_kps[RIGHT_HIP, 1]

    # 4. 関節角度
    features["angle_l_shoulder_hip_knee"] = _angle_between(
        kps[LEFT_SHOULDER], kps[LEFT_HIP], kps[LEFT_KNEE]
    )
    features["angle_r_shoulder_hip_knee"] = _angle_between(
        kps[RIGHT_SHOULDER], kps[RIGHT_HIP], kps[RIGHT_KNEE]
    )
    features["angle_l_hip_knee_ankle"] = _angle_between(
        kps[LEFT_HIP], kps[LEFT_KNEE], kps[LEFT_ANKLE]
    )
    features["angle_r_hip_knee_ankle"] = _angle_between(
        kps[RIGHT_HIP], kps[RIGHT_KNEE], kps[RIGHT_ANKLE]
    )

    # 5. hip相対y座標
    features["rel_left_hip_y"] = rel_kps[LEFT_HIP, 1]
    features["rel_right_hip_y"] = rel_kps[RIGHT_HIP, 1]

    # 6. knee相対y座標
    features["rel_left_knee_y"] = rel_kps[LEFT_KNEE, 1]
    features["rel_right_knee_y"] = rel_kps[RIGHT_KNEE, 1]

    # 7. 下半身キーポイントの信頼度
    lower_body_indices = [LEFT_HIP, RIGHT_HIP, LEFT_KNEE, RIGHT_KNEE, LEFT_ANKLE, RIGHT_ANKLE]
    features["mean_conf_lower"] = float(np.mean(scores[lower_body_indices]))
    features["min_conf_lower"] = float(np.min(scores[lower_body_indices]))
    features["conf_left_knee"] = scores[LEFT_KNEE]
    features["conf_right_knee"] = scores[RIGHT_KNEE]
    features["conf_left_ankle"] = scores[LEFT_ANKLE]
    features["conf_right_ankle"] = scores[RIGHT_ANKLE]

    # 8. 上半身-下半身の高さ比
    upper_y = np.mean([rel_kps[LEFT_SHOULDER, 1], rel_kps[RIGHT_SHOULDER, 1]])
    hip_y = np.mean([rel_kps[LEFT_HIP, 1], rel_kps[RIGHT_HIP, 1]])
    ankle_y = np.mean([rel_kps[LEFT_ANKLE, 1], rel_kps[RIGHT_ANKLE, 1]])
    torso_len = hip_y - upper_y
    leg_len = ankle_y - hip_y
    features["torso_leg_ratio"] = torso_len / (leg_len + 1e-8)

    # 9. knee水平位置（sitting時は前方に出やすい）
    features["left_knee_x_offset"] = rel_kps[LEFT_KNEE, 0] - rel_kps[LEFT_HIP, 0]
    features["right_knee_x_offset"] = rel_kps[RIGHT_KNEE, 0] - rel_kps[RIGHT_HIP, 0]

    return features


print("特徴量抽出関数を定義しました (22特徴量)")

特徴量抽出関数を定義しました (22特徴量)


In [3]:
# ============================================================
# Cell 4: データ読み込み
# ============================================================

def load_data():
    """JSONファイルを読み込み、特徴量とラベルを返す"""
    X_list = []
    y_list = []
    filenames = []
    raw_data = []  # ルールベース評価用

    for label_str, label_int in [("sitting", 0), ("standing", 1)]:
        folder = DATA_DIR / f"output_jsons_{label_str}"
        for fp in sorted(folder.glob("*.json")):
            with open(fp) as f:
                data = json.load(f)
            feats = extract_features(data)
            X_list.append(feats)
            y_list.append(label_int)
            filenames.append(fp.name)
            raw_data.append((data, label_str, fp.name))

    feature_names = list(X_list[0].keys())
    X = np.array([[row[k] for k in feature_names] for row in X_list])
    y = np.array(y_list)
    return X, y, feature_names, filenames, raw_data


X, y, feature_names, filenames, raw_data = load_data()

print(f"サンプル数: {X.shape[0]}")
print(f"特徴量数: {X.shape[1]}")
print(f"クラス分布: sitting={np.sum(y == 0)}, standing={np.sum(y == 1)}")
print(f"特徴量リスト: {feature_names}")

サンプル数: 927
特徴量数: 22
クラス分布: sitting=440, standing=487
特徴量リスト: ['bbox_aspect_ratio', 'left_hip_knee_vert', 'right_hip_knee_vert', 'left_hip_ankle_vert', 'right_hip_ankle_vert', 'angle_l_shoulder_hip_knee', 'angle_r_shoulder_hip_knee', 'angle_l_hip_knee_ankle', 'angle_r_hip_knee_ankle', 'rel_left_hip_y', 'rel_right_hip_y', 'rel_left_knee_y', 'rel_right_knee_y', 'mean_conf_lower', 'min_conf_lower', 'conf_left_knee', 'conf_right_knee', 'conf_left_ankle', 'conf_right_ankle', 'torso_leg_ratio', 'left_knee_x_offset', 'right_knee_x_offset']


In [4]:
# ============================================================
# Cell 5: ルールベース v1（再実装・バグあり）
# ============================================================
# SESSION_NOTESより: スコアリング方式、5特徴量
# バグ: aspect ratio の elif 閾値が 0.9 と小さすぎ
#       bh/bw が 0.9〜1.2 の範囲（多くの sitting）が
#       どの条件にも引っかからず standing に誤判定される

def rule_based_predict_v1(data: dict) -> str:
    """ルールベース v1（バグあり）: スコアリング方式、5特徴量

    バグ: aspect ratio の判定で elif 閾値が 0.9 と小さすぎる。
    bh/bw が 0.9〜1.2 の sitting サンプルがニュートラルのまま残り
    広い bbox の人物が standing に誤判定される。
    正しくは < 1.2 で sitting 判定すべき（v2 で修正）。
    """
    feats = extract_features(data)
    score = 0

    # aspect ratio の判定
    # BUG: elif の閾値が 0.9 と小さすぎる（正しくは < 1.2）
    # bh/bw が 0.9〜1.2 の sitting サンプルが score=0 のまま standing に誤判定
    bh_bw = feats["bbox_aspect_ratio"]
    if bh_bw > 2.0:            # 縦長 bbox → standing signal
        score += 2
    elif bh_bw < 0.9:          # BUG: 0.9 未満のみ sitting 判定（本来は 1.2 未満）
        score -= 2

    # hip-knee 垂直距離が両側とも小さい → sitting（膝が腰に近い）
    if feats["left_hip_knee_vert"] < 0.10 and feats["right_hip_knee_vert"] < 0.10:
        score -= 2

    # hip-knee-ankle 角度が両側とも小さい → sitting（膝が曲がっている）
    if feats["angle_l_hip_knee_ankle"] < 90 and feats["angle_r_hip_knee_ankle"] < 90:
        score -= 1

    return "sitting" if score < 0 else "standing"


# 全データで評価
y_true_str = [label for _, label, _ in raw_data]
y_pred_v1 = [rule_based_predict_v1(data) for data, _, _ in raw_data]

f1_v1 = f1_score(y_true_str, y_pred_v1, average="macro", pos_label=None)
print(f"=== ルールベース v1 Macro F1: {f1_v1:.4f} ===")
print("(記録値: 0.7172)\n")
print(classification_report(y_true_str, y_pred_v1))

=== ルールベース v1 Macro F1: 0.7173 ===
(記録値: 0.7172)

              precision    recall  f1-score   support

     sitting       0.71      0.69      0.70       440
    standing       0.73      0.74      0.74       487

    accuracy                           0.72       927
   macro avg       0.72      0.72      0.72       927
weighted avg       0.72      0.72      0.72       927



In [5]:
# ============================================================
# Cell 6: ルールベース v2（バグ修正 + 8特徴量）
# ============================================================
# SESSION_NOTESより: バグ修正 + 特徴量追加
# 追加: torso_leg_ratio, left/right_knee_x_offset, rel_left/right_hip_y
#
# データ実測値（各条件の発火率）:
#   aspect_ratio < 1.2   : sitting=43%, standing=15%  → sitting signal（バグ修正後）
#   hip_knee_vert < 0.125: sitting=93%, standing=62%  → sitting signal
#   angle < 110°         : sitting=67%, standing=45%  → weak sitting signal
#   torso_leg_ratio < 0.3: sitting= 3%, standing= 7%  → standing signal（脚が体幹より長い）
#   knee_x_offset > 0.20 : sitting=48%, standing=28%  → sitting signal（膝が横にずれる）
#   hip_y < 0.50         : sitting= 6%, standing=24%  → standing signal（腰が bbox 上部）

def rule_based_predict_v2(data: dict) -> str:
    """ルールベース v2（バグ修正済み）: 8特徴量

    v1 からの変更点:
    - aspect ratio の閾値を 1.2 に修正（0.9 → 1.2）
    - hip_knee_vert しきい値調整（0.10 → 0.125）
    - 追加: torso_leg_ratio（脚が長い → standing）
    - 追加: knee_x_offset（膝が横にずれている → sitting）
    - 追加: rel_hip_y（腰が bbox 上部にある → standing）
    """
    feats = extract_features(data)
    score = 0

    # aspect ratio（FIXED: 閾値を 1.2 に修正）
    bh_bw = feats["bbox_aspect_ratio"]
    if bh_bw < 1.2:        # 横長 bbox → sitting
        score -= 2
    elif bh_bw > 2.0:      # 縦長 bbox → standing
        score += 2

    # hip-knee 垂直距離（しきい値を 0.125 に調整）
    if feats["left_hip_knee_vert"] < 0.125 and feats["right_hip_knee_vert"] < 0.125:
        score -= 2

    # hip-knee-ankle 角度（110° 以下 = 膝の曲がりが大きい → sitting）
    if feats["angle_l_hip_knee_ankle"] < 110 and feats["angle_r_hip_knee_ankle"] < 110:
        score -= 1

    # torso_leg_ratio: 小さい（脚が体幹より長い）→ standing
    if feats["torso_leg_ratio"] < 0.30:
        score += 1

    # knee x-offset: 膝が腰より横にずれている → sitting（前後に出やすい）
    if abs(feats["left_knee_x_offset"]) > 0.20 or abs(feats["right_knee_x_offset"]) > 0.20:
        score -= 1

    # rel_hip_y: 腰が bbox の上部にある（小さい y 値）→ standing
    if feats["rel_left_hip_y"] < 0.50 or feats["rel_right_hip_y"] < 0.50:
        score += 1

    return "sitting" if score < 0 else "standing"


# 全データで評価
y_pred_v2 = [rule_based_predict_v2(data) for data, _, _ in raw_data]

f1_v2 = f1_score(y_true_str, y_pred_v2, average="macro", pos_label=None)
print(f"=== ルールベース v2 Macro F1: {f1_v2:.4f} ===")
print("(記録値: 0.7300)\n")
print(classification_report(y_true_str, y_pred_v2))

=== ルールベース v2 Macro F1: 0.7300 ===
(記録値: 0.7300)

              precision    recall  f1-score   support

     sitting       0.67      0.87      0.76       440
    standing       0.84      0.61      0.70       487

    accuracy                           0.73       927
   macro avg       0.75      0.74      0.73       927
weighted avg       0.76      0.73      0.73       927



In [6]:
# ============================================================
# Cell 7: LightGBM 学習 (src/train.py の内容)
# ============================================================

def train_and_evaluate():
    np.random.seed(SEED)
    print(f"データ: {X.shape[0]} samples, {X.shape[1]} features")
    print(f"クラス分布: sitting={np.sum(y == 0)}, standing={np.sum(y == 1)}")
    print(f"特徴量: {feature_names}\n")

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    oof_preds = np.zeros(len(y))
    fold_scores = []
    models = []

    params = {
        "objective": "binary",
        "metric": "binary_logloss",
        "verbosity": -1,
        "seed": SEED,
        "n_estimators": 300,
        "learning_rate": 0.05,
        "num_leaves": 31,
        "max_depth": 6,
        "min_child_samples": 10,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.1,
        "reg_lambda": 0.1,
    }

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.log_evaluation(0)],
        )

        val_pred = model.predict(X_val)
        oof_preds[val_idx] = val_pred
        fold_f1 = f1_score(y_val, val_pred, average="macro")
        fold_scores.append(fold_f1)
        models.append(model)
        print(f"Fold {fold}: Macro F1 = {fold_f1:.4f}")

    # 全体OOF結果
    overall_f1 = f1_score(y, oof_preds, average="macro")
    print(f"\n=== Overall OOF Macro F1: {overall_f1:.4f} ===")
    print(f"Fold scores: {[f'{s:.4f}' for s in fold_scores]}")
    print(f"Mean: {np.mean(fold_scores):.4f} +/- {np.std(fold_scores):.4f}\n")

    print("Classification Report:")
    print(classification_report(y, oof_preds, target_names=["sitting", "standing"]))

    print("Confusion Matrix:")
    print(confusion_matrix(y, oof_preds))

    # Feature importance
    print("\nFeature Importance (gain):")
    avg_importance = np.mean(
        [m.feature_importances_ for m in models], axis=0
    )
    sorted_idx = np.argsort(avg_importance)[::-1]
    for i in sorted_idx:
        print(f"  {feature_names[i]:35s} {avg_importance[i]:.1f}")

    # モデル保存
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    for fold, model in enumerate(models):
        model_path = RESULTS_DIR / f"lgbm_fold{fold}.pkl"
        with open(model_path, "wb") as f:
            pickle.dump(model, f)
    print(f"\nモデル保存先: {RESULTS_DIR}")

    return overall_f1, fold_scores, models, feature_names


overall_f1, fold_scores, lgbm_models, feat_names = train_and_evaluate()

データ: 927 samples, 22 features
クラス分布: sitting=440, standing=487
特徴量: ['bbox_aspect_ratio', 'left_hip_knee_vert', 'right_hip_knee_vert', 'left_hip_ankle_vert', 'right_hip_ankle_vert', 'angle_l_shoulder_hip_knee', 'angle_r_shoulder_hip_knee', 'angle_l_hip_knee_ankle', 'angle_r_hip_knee_ankle', 'rel_left_hip_y', 'rel_right_hip_y', 'rel_left_knee_y', 'rel_right_knee_y', 'mean_conf_lower', 'min_conf_lower', 'conf_left_knee', 'conf_right_knee', 'conf_left_ankle', 'conf_right_ankle', 'torso_leg_ratio', 'left_knee_x_offset', 'right_knee_x_offset']



/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 0: Macro F1 = 0.8004


/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 1: Macro F1 = 0.8226


...

Fold 2: Macro F1 = 0.7729


/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 3: Macro F1 = 0.7596


Fold 4: Macro F1 = 0.8000

=== Overall OOF Macro F1: 0.7915 ===
Fold scores: ['0.8004', '0.8226', '0.7729', '0.7596', '0.8000']
Mean: 0.7911 +/- 0.0223

Classification Report:
              precision    recall  f1-score   support

     sitting       0.77      0.80      0.78       440
    standing       0.81      0.79      0.80       487

    accuracy                           0.79       927
   macro avg       0.79      0.79      0.79       927
weighted avg       0.79      0.79      0.79       927

Confusion Matrix:
[[350  90]
 [103 384]]

Feature Importance (gain):
  bbox_aspect_ratio                   511.2
  left_knee_x_offset                  370.0
  angle_l_hip_knee_ankle              331.0
  angle_r_hip_knee_ankle              329.2
  right_knee_x_offset                 320.8
  left_hip_knee_vert                  315.8
  angle_r_shoulder_hip_knee           299.6
  right_hip_knee_vert                 269.2
  angle_l_shoulder_hip_knee           254.0
  rel_right_hip_y               

/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [7]:
# ============================================================
# Cell 8: 推論関数 (src/predict.py の内容)
# ============================================================

_MODELS_CACHE = None


def _load_models():
    """学習済みLightGBMモデルを遅延読み込み"""
    global _MODELS_CACHE
    if _MODELS_CACHE is not None:
        return _MODELS_CACHE
    _MODELS_CACHE = []
    for fold in range(5):
        model_path = RESULTS_DIR / f"lgbm_fold{fold}.pkl"
        with open(model_path, "rb") as f:
            _MODELS_CACHE.append(pickle.load(f))
    return _MODELS_CACHE


def _extract_features_for_predict(data: dict) -> np.ndarray:
    """入力dictから特徴量ベクトルを抽出する

    学習時と同じ順序・同じ計算で22個の特徴量を生成。
    特徴量はbbox正規化したkeypoint位置、関節角度、
    keypoint信頼度などで構成される。
    """
    bbox = data["bbox"]
    kps = np.array(data["keypoints"])    # (17, 2)
    scores = np.array(data["keypoint_scores"])  # (17,)
    bx, by, bw, bh = bbox["x"], bbox["y"], bbox["w"], bbox["h"]

    # bbox正規化キーポイント
    rel_kps = np.zeros_like(kps)
    rel_kps[:, 0] = (kps[:, 0] - bx) / (bw + 1e-8)
    rel_kps[:, 1] = (kps[:, 1] - by) / (bh + 1e-8)

    features = []

    # 1. bbox aspect ratio
    features.append(bh / (bw + 1e-8))

    # 2-3. hip-knee垂直距離
    features.append(rel_kps[LEFT_KNEE, 1] - rel_kps[LEFT_HIP, 1])
    features.append(rel_kps[RIGHT_KNEE, 1] - rel_kps[RIGHT_HIP, 1])

    # 4-5. hip-ankle垂直距離
    features.append(rel_kps[LEFT_ANKLE, 1] - rel_kps[LEFT_HIP, 1])
    features.append(rel_kps[RIGHT_ANKLE, 1] - rel_kps[RIGHT_HIP, 1])

    # 6-9. 関節角度
    features.append(_angle_between(kps[LEFT_SHOULDER], kps[LEFT_HIP], kps[LEFT_KNEE]))
    features.append(_angle_between(kps[RIGHT_SHOULDER], kps[RIGHT_HIP], kps[RIGHT_KNEE]))
    features.append(_angle_between(kps[LEFT_HIP], kps[LEFT_KNEE], kps[LEFT_ANKLE]))
    features.append(_angle_between(kps[RIGHT_HIP], kps[RIGHT_KNEE], kps[RIGHT_ANKLE]))

    # 10-11. hip相対y座標
    features.append(rel_kps[LEFT_HIP, 1])
    features.append(rel_kps[RIGHT_HIP, 1])

    # 12-13. knee相対y座標
    features.append(rel_kps[LEFT_KNEE, 1])
    features.append(rel_kps[RIGHT_KNEE, 1])

    # 14-15. 下半身keypoint信頼度
    lower_body = [LEFT_HIP, RIGHT_HIP, LEFT_KNEE, RIGHT_KNEE, LEFT_ANKLE, RIGHT_ANKLE]
    features.append(float(np.mean(scores[lower_body])))
    features.append(float(np.min(scores[lower_body])))

    # 16-19. 個別keypoint信頼度
    features.append(scores[LEFT_KNEE])
    features.append(scores[RIGHT_KNEE])
    features.append(scores[LEFT_ANKLE])
    features.append(scores[RIGHT_ANKLE])

    # 20. torso/leg比率
    upper_y = (rel_kps[LEFT_SHOULDER, 1] + rel_kps[RIGHT_SHOULDER, 1]) / 2
    hip_y = (rel_kps[LEFT_HIP, 1] + rel_kps[RIGHT_HIP, 1]) / 2
    ankle_y = (rel_kps[LEFT_ANKLE, 1] + rel_kps[RIGHT_ANKLE, 1]) / 2
    torso_len = hip_y - upper_y
    leg_len = ankle_y - hip_y
    features.append(torso_len / (leg_len + 1e-8))

    # 21-22. knee x方向オフセット
    features.append(rel_kps[LEFT_KNEE, 0] - rel_kps[LEFT_HIP, 0])
    features.append(rel_kps[RIGHT_KNEE, 0] - rel_kps[RIGHT_HIP, 0])

    return np.array(features).reshape(1, -1)


def sitting_prediction(data: dict) -> str:
    """人物が座っているか立っているかを判定する

    LightGBM 5-foldアンサンブルモデルを使用。
    各foldのモデルの予測確率を平均し、0.5を閾値として判定する。
    特徴量はbbox正規化keypoint位置、関節角度、信頼度スコアなど
    22次元のベクトルで構成される。

    Args:
        data: 以下のキーを持つdict
            - bbox: {"x": float, "y": float, "w": float, "h": float}
              人物を囲む矩形の左上頂点座標と幅・高さ（ピクセル）
            - bbox_confidence: float
              矩形推定の信頼度
            - keypoints: 17要素のリスト、各要素は[x, y]の2要素リスト
              COCO形式の関節点座標
            - keypoint_scores: 17要素のリスト
              各関節点の推定信頼度

    Returns:
        "sitting" - 座っていると判定された場合
        "standing" - 立っていると判定された場合
    """
    models = _load_models()
    X_feat = _extract_features_for_predict(data)

    # 5-foldモデルの予測確率を平均
    proba_sum = 0.0
    for model in models:
        proba = model.predict_proba(X_feat)[0, 1]  # standing確率
        proba_sum += proba
    avg_proba = proba_sum / len(models)

    # 0.5を閾値として判定（0=sitting, 1=standing）
    if avg_proba < 0.5:
        return "sitting"
    else:
        return "standing"


print("推論関数を定義しました")
print("sitting_prediction(data) -> 'sitting' or 'standing'")

推論関数を定義しました
sitting_prediction(data) -> 'sitting' or 'standing'


In [8]:
# ============================================================
# Cell 9: 評価 (src/evaluate.py の内容)
# ============================================================

def evaluate():
    """全データに対してsitting_predictionを実行し、Macro F1を計算"""
    y_true = []
    y_pred = []
    errors = []

    for label_str in ["sitting", "standing"]:
        folder = DATA_DIR / f"output_jsons_{label_str}"
        for fp in sorted(folder.glob("*.json")):
            with open(fp) as f:
                data = json.load(f)
            pred = sitting_prediction(data)
            y_true.append(label_str)
            y_pred.append(pred)
            if pred != label_str:
                errors.append((fp.name, label_str, pred))

    macro_f1 = f1_score(y_true, y_pred, average="macro", pos_label=None)
    print(f"=== Macro F1: {macro_f1:.4f} ===")
    print("(記録値: 0.7915)\n")
    print("Classification Report:")
    print(classification_report(y_true, y_pred))
    print("Confusion Matrix (rows=true, cols=pred):")
    print("           sitting  standing")
    cm = confusion_matrix(y_true, y_pred, labels=["sitting", "standing"])
    print(f"sitting    {cm[0][0]:7d}  {cm[0][1]:8d}")
    print(f"standing   {cm[1][0]:7d}  {cm[1][1]:8d}")

    print(f"\n誤分類数: {len(errors)} / {len(y_true)}")
    if errors:
        print("\n誤分類サンプル (先頭20件):")
        for fname, true, pred in errors[:20]:
            print(f"  {fname}: true={true}, pred={pred}")

    return macro_f1


lgbm_f1 = evaluate()

/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with

/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with

/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with

/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with

/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with

/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with

/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with

/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with

=== Macro F1: 0.9870 ===
(記録値: 0.7915)

Classification Report:
              precision    recall  f1-score   support

     sitting       0.98      1.00      0.99       440
    standing       1.00      0.98      0.99       487

    accuracy                           0.99       927
   macro avg       0.99      0.99      0.99       927
weighted avg       0.99      0.99      0.99       927

Confusion Matrix (rows=true, cols=pred):
           sitting  standing
sitting        438         2
standing        10       477

誤分類数: 12 / 927

誤分類サンプル (先頭20件):
  067532353.jpg.json: true=sitting, pred=standing
  082744574.jpg.json: true=sitting, pred=standing
  004663590.jpg.json: true=standing, pred=sitting
  010943040.jpg.json: true=standing, pred=sitting
  027046911.jpg.json: true=standing, pred=sitting
  043234774.jpg.json: true=standing, pred=sitting
  060051686.jpg.json: true=standing, pred=sitting
  066503667.jpg.json: true=standing, pred=sitting
  069363484.jpg.json: true=standing, pred=sittin

/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/estyle-155/Documents/work_sample_test_starterRepository/workspace/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with

# アプローチ比較サマリー

| アプローチ | 特徴量数 | CV Macro F1 | 改善幅 | 備考 |
|-----------|---------|-------------|--------|------|
| ルールベース v1 | 5 | ≈ 0.7172 | baseline | aspect ratio の if/elif 順バグあり |
| ルールベース v2 | 8 | ≈ 0.7300 | +0.013 | バグ修正 + 特徴量3つ追加 |
| LightGBM 5-fold | 22 | ≈ 0.7915 | +0.062 | StratifiedKFold, OOF評価 |

## 重要な知見

- **bbox aspect ratio が特徴量重要度 1 位** (gain≈511)。縦長 bbox → standing、横長 bbox → sitting
- **hip-knee-ankle 角度**も重要 (gain≈330): 膝の曲がり具合が決め手
- ルールベース(0.73) vs LightGBM(0.79): 約 6 ポイント差。非線形な組み合わせが効いている
- Cohen's d で最強だった hip-knee 垂直距離は LightGBM では中位（角度特徴量との共線性）

## 次のステップ

- [ ] 閾値最適化（0.5 以外でMacro F1 改善の余地）
- [ ] 全 17 keypoint の相対座標を直接入力
- [ ] keypoint 間ユークリッド距離
- [ ] confidence score の 17 次元をそのまま特徴量に
- [ ] XGBoost / CatBoost との比較
- [ ] 爆発案: keypoint confidence score だけで分類